# Feature engineering

In [1]:
import pandas as pd
import numpy as np
from sklearn.model_selection import train_test_split, StratifiedKFold
from sklearn.preprocessing import TargetEncoder, KBinsDiscretizer, OrdinalEncoder
from sklearn.compose import ColumnTransformer
from sklearn.pipeline import Pipeline

from heart_failure.config.config import (
    INTERIM_DATA_DIR, HEART_DISEASE, SEX, AGE, MAX_HR, OLDPEAK, FASTING_BS
)
from heart_failure.config.features import (
    TEST_SIZE,
    RANDOM_STATE,
    TE_CV,
    TE_FEATURES,
    BINARIZED_FEATURES,
    BINARY_CAT_FEATURES,
    Z_SCORED_FEATURES,
    OLDPEAK_TE_FEATURES,
    MAX_HR_TE_FEATURES,
    FASTING_BS_TE_FEATURES
)
from heart_failure.features import (
    GroupZScore, TargetEncoderByBins, CrossTargetEncoder, add_features_ratio
)

2026-06-14 14:36:37.438 | INFO     | heart_failure.config.config:<module>:11 - PROJ_ROOT path is: D:\heart_failure


In [2]:
df = pd.read_csv(INTERIM_DATA_DIR / "heart.csv")

y = df[HEART_DISEASE]
X = df.drop(columns=[HEART_DISEASE])

X_train, X_test, y_train, y_test = train_test_split(
    X,
    y,
    test_size=TEST_SIZE,
    random_state=RANDOM_STATE,
    stratify=df[[SEX, HEART_DISEASE]]
)

### Features

In [3]:
X_train = add_features_ratio(X_train, AGE, MAX_HR)
X_test = add_features_ratio(X_test, AGE, MAX_HR)

In [4]:
cv = StratifiedKFold(n_splits=TE_CV, shuffle=True, random_state=RANDOM_STATE)

feature_engineering = Pipeline([
    ("group_zscore", GroupZScore([AGE, SEX], Z_SCORED_FEATURES)),
    ("oldpeak_te", TargetEncoderByBins([OLDPEAK], OLDPEAK_TE_FEATURES, n_bins=4, cv=cv)),
    ("max_hr_te", TargetEncoderByBins([MAX_HR], MAX_HR_TE_FEATURES, n_bins=4, cv=cv)),
    ("fasting_bs_te", CrossTargetEncoder([[FASTING_BS], FASTING_BS_TE_FEATURES], cv=cv))
])

In [5]:
X_train = feature_engineering.fit_transform(X_train, y_train)
X_test = feature_engineering.transform(X_test)

In [ ]:
preprocessor = ColumnTransformer([
    ("target_encoder", TargetEncoder(cv=cv), TE_FEATURES),
    ("binarizer", KBinsDiscretizer(n_bins=10, encode="ordinal"), BINARIZED_FEATURES),
    ("binary_encoder", OrdinalEncoder(), BINARY_CAT_FEATURES)
], remainder="passthrough")
preprocessor.set_output(transform="pandas")

In [7]:
X_train = preprocessor.fit_transform(X_train, y_train)
X_test = preprocessor.transform(X_test)

In [8]:
feature_names = preprocessor.get_feature_names_out()
feature_names = [col.split("__", 1)[1] for col in feature_names]
X_train.columns = feature_names

In [9]:
X_train

,RestingECG,ChestPainType,ST_Slope,Age,Sex,FastingBS,ExerciseAngina,RestingBP,Cholesterol,MaxHR,...,Age_MaxHR_ratio,Cholesterol_z,MaxHR_z,Oldpeak_ST_Slope_te,Oldpeak_ExerciseAngina_te,Oldpeak_ChestPainType_te,MaxHR_RestingECG_te,FastingBS_ST_Slope_te,FastingBS_ChestPainType_te,FastingBS_ExerciseAngina_te
382,0.500154,0.784990,0.824971,1.0,1.0,0.0,1.0,115.0,236.0,145,...,0.296552,-0.171093,0.323348,0.868307,0.913555,0.915732,0.435145,0.771370,0.723544,0.804543
162,0.525940,0.153642,0.190614,2.0,1.0,0.0,0.0,160.0,263.0,174,...,0.270115,0.637056,1.449518,0.113732,0.252109,0.059411,0.301169,0.157656,0.121841,0.262830
174,0.525940,0.793146,0.827750,4.0,1.0,0.0,1.0,140.0,266.0,134,...,0.388060,0.359247,-0.158729,0.858209,0.908868,0.911070,0.625676,0.778606,0.736211,0.816800
154,0.660018,0.122457,0.199339,1.0,1.0,0.0,0.0,120.0,291.0,160,...,0.256250,1.100027,0.279401,0.110319,0.243637,0.061119,0.377081,0.158609,0.083884,0.249921
774,0.497331,0.353152,0.833365,0.0,1.0,0.0,1.0,120.0,231.0,182,...,0.208791,-0.172506,1.398594,0.891474,0.924130,0.424700,0.274971,0.770369,0.222191,0.817780
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
661,0.609699,0.802070,0.199339,1.0,1.0,0.0,0.0,110.0,197.0,177,...,0.248588,-0.777522,0.937099,0.110319,0.243637,0.603467,0.393129,0.158609,0.733288,0.249921
55,0.525940,0.153642,0.190614,3.0,0.0,0.0,0.0,160.0,194.0,170,...,0.300000,-1.132416,1.325792,0.113732,0.252109,0.059411,0.301169,0.157656,0.121841,0.262830
455,0.512936,0.353472,0.835116,7.0,1.0,0.0,1.0,120.0,233.0,80,...,0.762500,-0.297639,-2.002218,0.837591,0.657552,0.220884,0.748236,0.786345,0.315220,0.825675
188,0.497331,0.802070,0.833365,3.0,0.0,0.0,1.0,120.0,328.0,110,...,0.454545,1.608236,-1.207000,0.760590,0.843790,0.826873,0.738377,0.770369,0.733288,0.817780
